### OLS LINEAR REGRESSION

Consider 15 consecutive trading days in which we record the yield of a 2-year, 3-year, 5-year, and 10-year treasury bond

Let's investigate a linear regression for the yield of the 3-year bond in terms of the other yields

$$T_3 \sim \beta_0 + \beta_1 T_2 + \beta_2 T_5 + \beta_3 T_{10}$$

In [14]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "2-year": [
        4.69, 4.81, 4.81, 4.79, 4.79,
        4.83, 4.81, 4.81, 4.83, 4.81,
        4.82, 4.82, 4.80, 4.78, 4.79
    ],
    "3-year": [
        4.58, 4.71, 4.72, 4.78, 4.77,
        4.75, 4.71, 4.72, 4.76, 4.73,
        4.75, 4.75, 4.73, 4.71, 4.71
    ],
    "5-year": [
        4.57, 4.69, 4.70, 4.77, 4.77,
        4.73, 4.72, 4.74, 4.77, 4.75,
        4.77, 4.76, 4.75, 4.72, 4.71
    ],
    "10-year": [
        4.63, 4.73, 4.74, 4.81, 4.80,
        4.79, 4.76, 4.77, 4.80, 4.77,
        4.80, 4.80, 4.78, 4.73, 4.73
    ]
})

In [15]:
X = df[['2-year', '5-year', '10-year']]
X = sm.add_constant(X)

y = df['3-year']

model = sm.OLS(y, X)
results = model.fit()
results.summary()

RSS = np.sum(results.resid**2)

#### EXERCISE
1. Linear regression of JPM prices wrt prices of other stocks
2. Compute weekly percentage returns
3. Linear regression of JPM returns wrt returns of other stocks

In [16]:
data = pd.read_csv('./data/financials2012.csv')
data = data.rename(columns={'RY (RBC)': 'RY', 'BCS (Barclays)': 'BCS'})
data.head(10)


,Date,JPM,GS,MS,BAC,RBS,CS,UBS,RY,BCS
0,15-Oct-12,42.32,123.62,17.53,9.44,8.94,23.49,13.05,58.91,14.91
1,8-Oct-12,41.62,120.20,17.31,9.12,8.64,22.43,12.61,57.97,14.78
2,1-Oct-12,41.71,119.31,17.50,9.32,8.46,22.57,12.80,58.69,14.50
3,24-Sep-12,40.18,113.68,16.74,8.83,8.32,21.15,12.18,57.41,13.87
4,17-Sep-12,40.58,116.72,17.08,9.11,8.91,22.88,12.92,57.41,14.42
5,10-Sep-12,41.27,121.36,18.24,9.55,9.01,23.11,13.48,57.79,14.81
6,4-Sep-12,39.01,116.33,17.08,8.80,7.86,21.16,12.35,57.47,13.17
7,27-Aug-12,36.87,105.72,15.00,7.98,7.19,19.26,11.15,56.01,11.63
8,20-Aug-12,36.90,104.50,14.56,8.15,7.09,19.21,11.20,54.22,11.86
9,13-Aug-12,36.71,103.15,14.59,7.99,7.29,18.25,11.04,54.56,12.07


In [17]:
# 1. Linear regression of JPM prices
X = data[['GS', 'MS', 'BAC', 'RBS', 'CS', 'UBS', 'RY', 'BCS']]
y = data['JPM']

X = sm.add_constant(X)

model = sm.OLS(y, X)
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    JPM   R-squared:                       0.917
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     44.24
Date:                Thu, 16 Jul 2026   Prob (F-statistic):           3.79e-15
Time:                        16:53:27   Log-Likelihood:                -58.842
No. Observations:                  41   AIC:                             135.7
Df Residuals:                      32   BIC:                             151.1
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.8166      7.711      1.143      0.261      -6.890      24.523
GS             0.0166      0.090      0.185      0.855      -0.166       0.200
MS             0.0668      0.379      0.176      0.861      -0.706       0.839
BAC            1.4713      0.426      3.453      0.002       0.603       2.339
RBS            1.2074      0.926      1.304      0.201      -0.678       3.093
CS             0.8999      0.235      3.827      0.001       0.421       1.379
UBS           -2.5373      0.862     -2.945      0.006      -4.292      -0.782
RY             0.3502      0.192      1.821      0.078      -0.041       0.742
BCS           -0.1705      0.469     -0.364      0.718      -1.125       0.784
==============================================================================
Omnibus:                       11.722   Durbin-Watson:                   1.257
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               11.962
Skew:                           1.035   Prob(JB):                      0.00253
Kurtosis:                       4.648   Cond. No.                     5.45e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 5.45e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [18]:
# 2. Compute weekly percentage returns
data["Date"] = pd.to_datetime(data["Date"], format="%d-%b-%y")
data = data.sort_values(by="Date", ascending=True).reset_index(drop=True)
data.head(10)

for col in data.columns[1:]:
    # data[f'{col}_PctReturn'] = data[f'{col}'].transform(lambda x: (x - x.shift(1)) / x.shift(1))
    data[f'{col}_PctReturn'] = data[f'{col}'].pct_change()
data

,Date,JPM,GS,MS,BAC,RBS,CS,UBS,RY,BCS,JPM_PctReturn,GS_PctReturn,MS_PctReturn,BAC_PctReturn,RBS_PctReturn,CS_PctReturn,UBS_PctReturn,RY_PctReturn,BCS_PctReturn
0,2012-01-11,35.13,97.76,16.47,6.58,7.36,21.83,11.83,49.32,12.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-17,36.54,107.42,18.22,7.04,8.59,25.12,13.58,51.50,13.64,0.040137,0.098813,0.106254,0.069909,0.167120,0.150710,0.147929,0.044201,0.127273
2,2012-01-23,36.39,110.42,18.43,7.26,8.71,25.83,13.93,51.25,13.78,-0.004105,0.027928,0.011526,0.031250,0.013970,0.028264,0.025773,-0.004854,0.010264
3,2012-01-30,37.44,116.11,20.17,7.81,9.09,26.97,14.54,52.52,14.77,0.028854,0.051531,0.094411,0.075758,0.043628,0.044135,0.043790,0.024780,0.071843
4,2012-02-06,36.78,112.74,19.53,8.04,8.77,24.90,13.78,52.34,14.49,-0.017628,-0.029024,-0.031730,0.029449,-0.035204,-0.076752,-0.052270,-0.003427,-0.018957
5,2012-02-13,37.63,114.51,19.03,7.99,8.85,25.95,14.20,52.27,15.51,0.023110,0.015700,-0.025602,-0.006219,0.009122,0.042169,0.030479,-0.001337,0.070393
6,2012-02-21,37.44,114.47,18.37,7.85,9.04,26.76,14.17,53.50,15.50,-0.005049,-0.000349,-0.034682,-0.017522,0.021469,0.031214,-0.002113,0.023532,-0.000645
7,2012-02-27,39.74,118.87,18.74,8.11,8.83,26.39,13.74,55.87,16.00,0.061432,0.038438,0.020142,0.033121,-0.023230,-0.013827,-0.030346,0.044299,0.032258
8,2012-03-05,40.13,116.22,18.25,8.03,8.24,25.59,13.35,56.15,15.00,0.009814,-0.022293,-0.026147,-0.009864,-0.066818,-0.030315,-0.028384,0.005012,-0.062500
9,2012-03-12,43.59,121.81,19.40,9.77,8.99,28.57,14.31,57.35,16.00,0.086220,0.048098,0.063014,0.216687,0.091019,0.116452,0.071910,0.021371,0.066667


In [20]:
# 3. Linear regression of JPM returns wrt returns of other stocks
X = data[['GS_PctReturn', 'MS_PctReturn', 'BAC_PctReturn', 'RBS_PctReturn', 'CS_PctReturn', 'UBS_PctReturn', 'RY_PctReturn', 'BCS_PctReturn']].dropna()
X = sm.add_constant(X)
y = data['JPM_PctReturn'].dropna()

model2 = sm.OLS(y, X)
results2 = model2.fit()
results2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          JPM_PctReturn   R-squared:                       0.774
Model:                            OLS   Adj. R-squared:                  0.715
Method:                 Least Squares   F-statistic:                     13.24
Date:                Thu, 16 Jul 2026   Prob (F-statistic):           4.34e-08
Time:                        16:55:26   Log-Likelihood:                 98.798
No. Observations:                  40   AIC:                            -179.6
Df Residuals:                      31   BIC:                            -164.4
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.0034      0.004     -0.836      0.410      -0.012       0.005
GS_PctReturn      0.7663      0.234      3.272      0.003       0.289       1.244
MS_PctReturn     -0.0778      0.179     -0.435      0.667      -0.443       0.287
BAC_PctReturn     0.2980      0.117      2.543      0.016       0.059       0.537
RBS_PctReturn     0.2812      0.137      2.054      0.048       0.002       0.560
CS_PctReturn     -0.0576      0.146     -0.395      0.695      -0.355       0.240
UBS_PctReturn    -0.4140      0.233     -1.779      0.085      -0.889       0.061
RY_PctReturn      0.1476      0.213      0.693      0.493      -0.287       0.582
BCS_PctReturn    -0.0088      0.121     -0.073      0.943      -0.256       0.238
==============================================================================
Omnibus:                        0.997   Durbin-Watson:                   2.281
Prob(Omnibus):                  0.608   Jarque-Bera (JB):                0.958
Skew:                           0.345   Prob(JB):                        0.619
Kurtosis:                       2.686   Cond. No.                         77.4
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""